In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Create 30 days of hourly timestamps (720 hours)
timestamps = pd.date_range(start='2026-01-01 00:00', periods=720, freq='h')

stations = ['STN_Noida_01', 'STN_Faridabad_02', 'STN_Gurgaon_03', 'STN_Delhi_04']
chargers_per_station = ['CHG_A', 'CHG_B', 'CHG_C']

rows = []
for ts in timestamps:
    hour = ts.hour
    is_weekend = 1 if ts.dayofweek >= 5 else 0

    # Peak charging windows: morning (8-10 AM) and evening (6-10 PM)
    peak_factor = 0.0
    if 8 <= hour <= 10:
        peak_factor = 0.35
    elif 18 <= hour <= 22:
        peak_factor = 0.45
    elif 0 <= hour <= 5:
        peak_factor = -0.20

    for stn in stations:
        stn_bias = {
            'STN_Noida_01': 0.14,
            'STN_Faridabad_02': 0.05,
            'STN_Gurgaon_03': 0.18,
            'STN_Delhi_04': -0.06
        }[stn]

        for chg in chargers_per_station:
            chg_bias = {'CHG_A': 0.06, 'CHG_B': 0.0, 'CHG_C': -0.08}[chg]
            raw_util = 0.38 + stn_bias + chg_bias + peak_factor + (0.04 * is_weekend) + np.random.normal(0, 0.08)
            util_rate = float(np.clip(raw_util, 0.05, 0.98))

            active_mins = round(util_rate * 60, 1)
            idle_mins = round(60 - active_mins, 1)
            energy_kwh = round(util_rate * 22.0, 2)  # e.g., 22 kW charger capacity
            is_bottleneck = 1 if util_rate >= 0.80 else 0

            rows.append({
                'Timestamp': ts,
                'Date': ts.date(),
                'Hour': hour,
                'Day_Name': ts.day_name(),
                'Is_Weekend': is_weekend,
                'Station_ID': stn,
                'Charger_ID': f"{stn}_{chg}",
                'Utilisation_Rate': round(util_rate, 4),
                'Active_Mins': active_mins,
                'Idle_Mins': idle_mins,
                'Energy_Delivered_kWh': energy_kwh,
                'Bottleneck_Flag': is_bottleneck
            })

df = pd.DataFrame(rows)
df.to_csv('charger_hourly_utilisation.csv', index=False)

print("Saved 'charger_hourly_utilisation.csv'! Shape:", df.shape)
print(df.head())

C:\Users\Diya\AppData\Local\Temp\ipykernel_5404\2562585594.py:15: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  is_weekend = 1 if ts.dayofweek >= 5 else 0


Saved 'charger_hourly_utilisation.csv'! Shape: (8640, 12)
   Timestamp        Date  Hour  Day_Name  Is_Weekend        Station_ID  \
0 2026-01-01  2026-01-01     0  Thursday           0      STN_Noida_01   
1 2026-01-01  2026-01-01     0  Thursday           0      STN_Noida_01   
2 2026-01-01  2026-01-01     0  Thursday           0      STN_Noida_01   
3 2026-01-01  2026-01-01     0  Thursday           0  STN_Faridabad_02   
4 2026-01-01  2026-01-01     0  Thursday           0  STN_Faridabad_02   

               Charger_ID  Utilisation_Rate  Active_Mins  Idle_Mins  \
0      STN_Noida_01_CHG_A            0.4197         25.2       34.8   
1      STN_Noida_01_CHG_B            0.3089         18.5       41.5   
2      STN_Noida_01_CHG_C            0.2918         17.5       42.5   
3  STN_Faridabad_02_CHG_A            0.4118         24.7       35.3   
4  STN_Faridabad_02_CHG_B            0.2113         12.7       47.3   

   Energy_Delivered_kWh  Bottleneck_Flag  
0                  9.23    

In [2]:
# 1. Idle-Time Analysis by Station
idle_by_station = df.groupby('Station_ID', as_index=False).agg(
    avg_utilisation_pct=('Utilisation_Rate', lambda s: round(s.mean() * 100, 2)),
    avg_idle_mins_per_hr=('Idle_Mins', 'mean'),
    total_idle_hours=('Idle_Mins', lambda s: round(s.sum() / 60, 1)),
    total_energy_kwh=('Energy_Delivered_kWh', 'sum'),
    bottleneck_hours=('Bottleneck_Flag', 'sum')
).round(2)

idle_by_station = idle_by_station.sort_values('avg_utilisation_pct', ascending=False).reset_index(drop=True)
print("=== 1. STATION SUMMARY & IDLE TIME ===")
print(idle_by_station)

# 2. Peak-Load Analysis by Hour (0-23)
hourly_load = df.groupby('Hour', as_index=False).agg(
    avg_utilisation_pct=('Utilisation_Rate', lambda s: round(s.mean() * 100, 2)),
    avg_idle_mins=('Idle_Mins', 'mean'),
    total_energy_kwh=('Energy_Delivered_kWh', 'sum'),
    bottleneck_count=('Bottleneck_Flag', 'sum')
).round(2)

peak_hours = hourly_load.sort_values('avg_utilisation_pct', ascending=False).head(5)
print("\n=== 2. TOP 5 PEAK UTILISATION HOURS ===")
print(peak_hours)

# 3. High / Low Performing Charger Ranking
charger_rank = df.groupby(['Charger_ID', 'Station_ID'], as_index=False).agg(
    avg_utilisation_pct=('Utilisation_Rate', lambda s: round(s.mean() * 100, 2)),
    avg_idle_mins_per_hr=('Idle_Mins', 'mean'),
    total_energy_kwh=('Energy_Delivered_kWh', 'sum'),
    bottleneck_hours=('Bottleneck_Flag', 'sum')
).round(2)

charger_rank = charger_rank.sort_values('avg_utilisation_pct', ascending=False).reset_index(drop=True)
charger_rank['Rank'] = charger_rank.index + 1
charger_rank['Performance_Tier'] = np.where(
    charger_rank['avg_utilisation_pct'] >= charger_rank['avg_utilisation_pct'].quantile(0.67), 'High Performing',
    np.where(charger_rank['avg_utilisation_pct'] <= charger_rank['avg_utilisation_pct'].quantile(0.33), 'Low Performing', 'Moderate')
)

print("\n=== 3. CHARGER RANKING (TOP 5 & BOTTOM 5) ===")
print("Top 5:")
print(charger_rank[['Rank', 'Charger_ID', 'avg_utilisation_pct', 'avg_idle_mins_per_hr', 'Performance_Tier']].head(5))
print("\nBottom 5:")
print(charger_rank[['Rank', 'Charger_ID', 'avg_utilisation_pct', 'avg_idle_mins_per_hr', 'Performance_Tier']].tail(5))

# 4. Utilisation Heatmap Matrix (Station x Hour)
heatmap_df = df.pivot_table(
    index='Station_ID',
    columns='Hour',
    values='Utilisation_Rate',
    aggfunc='mean'
).mul(100).round(1)

print("\n=== 4. UTILISATION HEATMAP MATRIX (STATION x HOUR %) ===")
print(heatmap_df.iloc[:, :12])  # First 12 hours (0-11)
print(heatmap_df.iloc[:, 12:]) # Remaining 12 hours (12-23)

# Save summary tables for Power BI
idle_by_station.to_csv('station_utilisation_summary.csv', index=False)
hourly_load.to_csv('hourly_peak_load.csv', index=False)
charger_rank.to_csv('charger_performance_ranking.csv', index=False)
print("\nSaved 'station_utilisation_summary.csv', 'hourly_peak_load.csv', and 'charger_performance_ranking.csv'!")

=== 1. STATION SUMMARY & IDLE TIME ===
         Station_ID  avg_utilisation_pct  avg_idle_mins_per_hr  \
0    STN_Gurgaon_03                63.64                 21.82   
1      STN_Noida_01                60.35                 23.79   
2  STN_Faridabad_02                52.04                 28.78   
3      STN_Delhi_04                41.24                 35.25   

   total_idle_hours  total_energy_kwh  bottleneck_hours  
0             785.5          30239.80               677  
1             856.4          28678.32               639  
2            1035.9          24729.32               486  
3            1269.2          19597.37               191  

=== 2. TOP 5 PEAK UTILISATION HOURS ===
    Hour  avg_utilisation_pct  avg_idle_mins  total_energy_kwh  \
19    19                88.62           6.83           7018.66   
18    18                88.54           6.88           7012.01   
20    20                88.53           6.88           7011.90   
21    21                88.39      